In [0]:
%run  ../notebooks/config_Parms


# Silver Layer 


Create Delta table if not exist as a target 

In [0]:
# Step 1: Create silver table if it doesn't exist yet
# This creates a Delta table with the specified schema.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.ibm
    (
        Date DATE,
        Open DOUBLE,
        High DOUBLE,
        Low DOUBLE,
        Close DOUBLE,
        Volume BIGINT,
        Last_updated TIMESTAMP,
        ingested_at TIMESTAMP
    )
    USING DELTA
""")

### Silver Transformations: 
- Reads the IBM bronze table, converts it to a Pandas DataFrame, 
- groups by 'Date' to get the latest records,
- applies data quality checks (nulls, types, ranges, regex, referential integrity, deduplication),
- renforce schems explicitly 
- and creates or replaces a temporary view named 'silver_ibm'.


In [0]:

from pyspark.sql.functions import col, to_date, to_timestamp, current_timestamp
from pyspark.sql.types import DoubleType, LongType

# Step 1: Read bronze — already a Spark DataFrame
source_raw = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.ibm")

# Step 2: Cast columns
source_casted = (
    source_raw
    .withColumn("Date",         to_date(col("Date"), "yyyy-MM-dd"))
    .withColumn("Open",         col("Open").cast(DoubleType()))
    .withColumn("High",         col("High").cast(DoubleType()))
    .withColumn("Low",          col("Low").cast(DoubleType()))
    .withColumn("Close",        col("Close").cast(DoubleType()))
    .withColumn("Volume",       col("Volume").cast(LongType()))
    .withColumn("Last_updated", to_timestamp(col("Last_updated")))
    .withColumn("ingested_at",  current_timestamp())
)

# Step 3: Drop nulls on critical columns
source_no_nulls = source_casted.dropna(subset=["Date", "Close", "Open", "High", "Low", "Volume"])

# Step 4: Deduplicate on merge key
source_deduped = source_no_nulls.dropDuplicates(["Date"])
# filter Data
source_filtered =source_deduped.filter(col("Close")>0)
# Step 5: Assign to source_clean for downstream use
source_clean = source_deduped

# Step 6: Preview
display(source_clean)

ALTER TABLE workspace.silver.ibm ADD COLUMN ingested_at TIMESTAMP

In [0]:
# Step 6: Merge cleaned data into the silver Delta table.
# This upserts records from source_clean into the target Delta table (catalog.silver_schema.ibm).
# - If a row with the same Date exists, it updates all columns.
# - If not, it inserts the new row.
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, f"{catalog}.{silver_schema}.ibm")

(
    target.alias("target").merge(
        source=source_clean.alias("source"),
        condition="target.Date = source.Date"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
%sql
SELECT * FROM workspace.silver.ibm order by date desc 

describe  history workspace.silver.ibm

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.ibm")